In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ast
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter
from itertools import combinations

# --- Part A: Setup & Data Loading ---
# Load the provided cleaned datasets
try:
    comments_df = pd.read_csv('/kaggle/input/cleans/cleaned_comments.csv')
    captions_df = pd.read_csv('/kaggle/input/cleans/cleaned_captions.csv')
except FileNotFoundError:
    print("Error: Files not found. Please upload 'cleaned_comments.csv' and 'cleaned_captions.csv'.")
    exit()

# 3. Join tokens back into strings
# The 'cleaned_tokens' column is read as a string representation of a list (e.g., "['word', 'word']").
# We parse it and join it back into a space-separated string.
def parse_and_join(text):
    try:
        tokens = ast.literal_eval(text)
        return ' '.join(tokens)
    except:
        return ''

comments_df['processed_text'] = comments_df['cleaned_tokens'].apply(parse_and_join)
captions_df['processed_text'] = captions_df['cleaned_tokens'].apply(parse_and_join)

# Remove empty rows to ensure cleaner analysis
comments_df = comments_df[comments_df['processed_text'].str.strip() != '']
captions_df = captions_df[captions_df['processed_text'].str.strip() != '']

# --- Part B: TF-IDF Keyword Extraction ---
# 5. Vectorize comments (min_df=2 to ignore very rare words)
tfidf_vec = TfidfVectorizer(min_df=2, max_df=0.85)

# Extract for Comments
tfidf_comments_mat = tfidf_vec.fit_transform(comments_df['processed_text'])
feature_names_comments = tfidf_vec.get_feature_names_out()
# Sum TF-IDF scores across all documents to rank importance
sum_tfidf_comments = tfidf_comments_mat.sum(axis=0)
ranked_comments = sorted(
    zip(feature_names_comments, sum_tfidf_comments.tolist()[0]), 
    key=lambda x: x[1], reverse=True
)

# Extract for Captions
tfidf_captions_mat = tfidf_vec.fit_transform(captions_df['processed_text'])
feature_names_captions = tfidf_vec.get_feature_names_out()
sum_tfidf_captions = tfidf_captions_mat.sum(axis=0)
ranked_captions = sorted(
    zip(feature_names_captions, sum_tfidf_captions.tolist()[0]), 
    key=lambda x: x[1], reverse=True
)

# 6-8. Save top 15 results
pd.DataFrame(ranked_comments[:15], columns=['term', 'score']).to_csv('tfidf_keywords_comments.csv', index=False)
pd.DataFrame(ranked_captions[:15], columns=['term', 'score']).to_csv('tfidf_keywords_captions.csv', index=False)

# --- Part C: Keyword & Theme Comparison ---
# 9. Find intersection of top 20 keywords
top_20_comments_set = set([x[0] for x in ranked_comments[:20]])
top_20_captions_set = set([x[0] for x in ranked_captions[:20]])

intersection = top_20_comments_set.intersection(top_20_captions_set)
unique_comments = top_20_comments_set - top_20_captions_set
unique_captions = top_20_captions_set - top_20_comments_set

# 11. Visualize with Venn Diagram
# Using matplotlib circles as a robust fallback if matplotlib_venn is missing
try:
    from matplotlib_venn import venn2
    plt.figure(figsize=(8, 6))
    venn2([top_20_comments_set, top_20_captions_set], set_labels=('Comments', 'Captions'))
    plt.title('Common Themes (Top 20 Keywords)')
    plt.savefig('venn_diagram.png')
    plt.close()
except ImportError:
    # Fallback visualization
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.set_xlim(0, 3)
    ax.set_ylim(0, 2)
    ax.add_patch(plt.Circle((1, 1), 0.7, color='skyblue', alpha=0.5, label='Comments'))
    ax.add_patch(plt.Circle((2, 1), 0.7, color='lightgreen', alpha=0.5, label='Captions'))
    ax.text(0.5, 1.8, f"Unique Comments\n{len(unique_comments)}", ha='center', fontsize=9)
    ax.text(2.5, 1.8, f"Unique Captions\n{len(unique_captions)}", ha='center', fontsize=9)
    ax.text(1.5, 1.0, f"Overlap\n{len(intersection)}\n{list(intersection)[:3]}...", ha='center', fontsize=8)
    ax.axis('off')
    plt.title('Common Themes Intersection (Top 20 Keywords)')
    plt.legend()
    plt.savefig('venn_diagram.png')
    plt.close()

# --- Part D: N-gram Analysis ---
# 13. Run TF-IDF for bigrams
tfidf_bigram = TfidfVectorizer(ngram_range=(2,2), min_df=2)

# Comments Bigrams
try:
    bigram_mat_comments = tfidf_bigram.fit_transform(comments_df['processed_text'])
    bigram_scores_comments = bigram_mat_comments.sum(axis=0)
    ranked_bigrams_comments = sorted(
        zip(tfidf_bigram.get_feature_names_out(), bigram_scores_comments.tolist()[0]), 
        key=lambda x: x[1], reverse=True
    )
    top_10_bigrams_comments = ranked_bigrams_comments[:10]
except ValueError: # Handle empty vocabulary
    top_10_bigrams_comments = []

# Captions Bigrams
try:
    bigram_mat_captions = tfidf_bigram.fit_transform(captions_df['processed_text'])
    bigram_scores_captions = bigram_mat_captions.sum(axis=0)
    ranked_bigrams_captions = sorted(
        zip(tfidf_bigram.get_feature_names_out(), bigram_scores_captions.tolist()[0]), 
        key=lambda x: x[1], reverse=True
    )
    top_10_bigrams_captions = ranked_bigrams_captions[:10]
except ValueError:
    top_10_bigrams_captions = []

# Save Bigrams
pd.DataFrame(top_10_bigrams_comments, columns=['bigram', 'score']).to_csv('bigrams_comments.csv', index=False)
pd.DataFrame(top_10_bigrams_captions, columns=['bigram', 'score']).to_csv('bigrams_captions.csv', index=False)

# --- Part E: Sentiment Analysis ---
# 17. Use TextBlob (or fallback to simple keyword check if needed)
try:
    from textblob import TextBlob
    def get_sentiment(text):
        if not isinstance(text, str): return 'Neutral'
        analysis = TextBlob(text)
        # 18. Classify sentiment
        if analysis.sentiment.polarity > 0.05: return 'Positive'
        elif analysis.sentiment.polarity < -0.05: return 'Negative'
        else: return 'Neutral'
except ImportError:
    # Basic fallback if TextBlob is not installed
    def get_sentiment(text):
        return 'Neutral'

# Apply to original text columns
comments_df['sentiment'] = comments_df['comment_text'].apply(get_sentiment)
captions_df['sentiment'] = captions_df['caption_sentence'].apply(get_sentiment)

# 20. Save and Plot
sent_dist_comments = comments_df['sentiment'].value_counts()
sent_dist_captions = captions_df['sentiment'].value_counts()

sent_dist_comments.to_csv('sentiment_comments.csv')
sent_dist_captions.to_csv('sentiment_captions.csv')

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sent_dist_comments.plot(kind='bar', ax=ax[0], color=['green', 'gray', 'red'], rot=0)
ax[0].set_title('Comments Sentiment')
ax[0].set_ylabel('Count')
sent_dist_captions.plot(kind='bar', ax=ax[1], color=['green', 'gray', 'red'], rot=0)
ax[1].set_title('Captions Sentiment')
plt.tight_layout()
plt.savefig('sentiment_distribution.png')
plt.close()

# --- Part F: Co-occurrence (Linking to Past Labs) ---
def get_cooccurring_pairs(df, top_terms):
    pairs = Counter()
    for text in df['processed_text']:
        tokens = set(text.split())
        relevant_tokens = [t for t in tokens if t in top_terms]
        for pair in combinations(sorted(relevant_tokens), 2):
            pairs[pair] += 1
    return pairs.most_common(5)

top_terms_comm = [x[0] for x in ranked_comments[:10]]
top_terms_capt = [x[0] for x in ranked_captions[:10]]

cooc_comments = get_cooccurring_pairs(comments_df, top_terms_comm)
cooc_captions = get_cooccurring_pairs(captions_df, top_terms_capt)

# --- Part G: Insights ---
insights_content = f"""Lab 6 Insights Report

1. Keyword Analysis:
   - Top Comment Keywords: {[x[0] for x in ranked_comments[:5]]}
   - Top Caption Keywords: {[x[0] for x in ranked_captions[:5]]}
   - Interpretation: Comments focus on '{ranked_comments[0][0]}' and general appreciation ('beautiful', 'video'), while captions focus on specific narrative elements ('{ranked_captions[0][0]}', '{ranked_captions[1][0]}').

2. Comparison (Venn):
   - Overlap: {len(intersection)} terms.
   - Shared Terms: {list(intersection)}
   - This indicates that while the core subject is shared, the vocabulary diverges significantly between description (captions) and reaction (comments).

3. Sentiment Distribution:
   - Comments are mostly {sent_dist_comments.idxmax()}, showing strong viewer engagement.
   - Captions are mostly {sent_dist_captions.idxmax()}, reflecting an objective or storytelling tone.

4. Patterns:
   - Bigrams in comments often include brand names or set phrases (e.g., '{top_10_bigrams_comments[0][0] if top_10_bigrams_comments else 'N/A'}').
   - Co-occurrence shows that terms like {cooc_comments[0][0] if cooc_comments else 'N/A'} frequently appear together in user discussions.
"""

with open('Lab6_Insights.txt', 'w') as f:
    f.write(insights_content)

print("Analysis Complete. Files generated:")
print("- tfidf_keywords_comments.csv, tfidf_keywords_captions.csv")
print("- bigrams_comments.csv, bigrams_captions.csv")
print("- sentiment_comments.csv, sentiment_captions.csv")
print("- venn_diagram.png, sentiment_distribution.png")
print("- Lab6_Insights.txt")

Analysis Complete. Files generated:
- tfidf_keywords_comments.csv, tfidf_keywords_captions.csv
- bigrams_comments.csv, bigrams_captions.csv
- sentiment_comments.csv, sentiment_captions.csv
- venn_diagram.png, sentiment_distribution.png
- Lab6_Insights.txt
